# reparameterization-trick — worked example 2: Show That Gradients Reach mu and logsigma Through the Reparam Trick

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reparameterization-trick`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The reparameterization trick makes sampling differentiable by expressing `z = mu + sigma * eps` where `eps` is treated as a constant. After calling `loss.backward()`, both `mu.grad` and `logsigma.grad` are populated. The gradient of a sum-loss with respect to `mu` is all-ones (since `d(z_i)/d(mu_i) = 1`), confirming that the reparameterized sample behaves as a deterministic function of the parameters.

## Worked solution

**Step 1 — create leaf tensors with `requires_grad=True`.** `mu` and `logsigma` need gradients because they are the parameters we want to update.

**Step 2 — fix `eps` as a constant.** Draw `eps = t.randn_like(mu).detach()` — detaching ensures PyTorch treats it as a constant, not a learnable input. In practice the VAE encoder creates `mu` and `logsigma` from input data; `eps` is fresh noise.

**Step 3 — compute `z` and a scalar loss.** `z = mu + (0.5*logsigma).exp() * eps`, then `loss = z.sum()`. The loss is scalar, needed for `.backward()`.

**Step 4 — call `loss.backward()`.** Gradients flow back through `z` to both `mu` and `logsigma`.

**Step 5 — verify gradients.** `mu.grad` should be all-ones. `logsigma.grad` should be `0.5 * eps * sigma` (by the chain rule through `exp`).

In [ ]:
import torch as t

t.manual_seed(42)
B, L = 2, 4

mu       = t.zeros(B, L, requires_grad=True)
logsigma = t.zeros(B, L, requires_grad=True)

# Fix eps as a constant (detached from the graph)
t.manual_seed(7)
eps = t.randn(B, L).detach()

# Reparameterized sample
sigma = (0.5 * logsigma).exp()
z = mu + sigma * eps
loss = z.sum()
loss.backward()

print('mu.grad (expect all 1.0):')
print(mu.grad)

# Analytical gradient: d(sum z)/d(logsigma_i) = 0.5 * sigma_i * eps_i
expected_logsigma_grad = 0.5 * sigma.detach() * eps
print('\nlogsigma.grad:')
print(logsigma.grad)
print('Expected:')
print(expected_logsigma_grad)
print('\nMatch:', t.allclose(logsigma.grad, expected_logsigma_grad, atol=1e-6))